# Paper Figures: Figure 4 - Sigmoidal Transitions in Cluster Probability

This notebook generates publication-ready figures for cluster probability transitions using sigmoidal fitting based on data assembled by `src/assemble_all_data.py`.

**Figure 4: Cluster Transitions** — Sigmoidal fits to cluster probability curves showing how animals transition between neural response clusters across trials.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Register dill/pathlib compatibility shim BEFORE importing dill
sys.path.insert(0, str(Path("../src").resolve()))
from pickle_compat import enable_dill_pathlib_compat
enable_dill_pathlib_compat()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import dill
from scipy.optimize import curve_fit

from figure_config import (
    configure_matplotlib, COLORS, HEATMAP_CMAP_DIV,
    DATAFOLDER, RESULTSFOLDER, FIGSFOLDER,
    SAVE_FIGS
)
from figure_plotting import (
    save_figure_atomic, scale_vlim_to_data
)

# Configure matplotlib
configure_matplotlib()
colors = COLORS  # Use shared color palette
custom_cmap = HEATMAP_CMAP_DIV  # Use shared colormap

## Load Assembled Data

Load the complete dataset from the pickle file generated by the assembly script.

In [ ]:
assembled_data_path = DATAFOLDER / "assembled_data.pickle"

with open(assembled_data_path, "rb") as f:
    data = dill.load(f)

# Extract main components
x_array = data["x_array"]
snips_photo = data["snips_photo"]
snips_angvel = data["snips_angvel"]
fits_df = data["fits_df"]
metadata = data.get("metadata", {})

print(f"Loaded assembled data from {assembled_data_path}")
print(f"\nData structure:")
print(f"  - x_array shape: {x_array.shape}")
print(f"  - snips_photo shape: {snips_photo.shape}")
print(f"  - x_array columns: {x_array.columns.tolist()}")
print(f"  - Number of trials: {len(x_array)}")
print(f"  - Clusters: {sorted(x_array.cluster_photo.unique())}")

print(f"\nFits DataFrame info:")
print(f"  - fits_df shape: {fits_df.shape if fits_df is not None else 'None'}")
print(f"  - fits_df columns: {fits_df.columns.tolist() if fits_df is not None else 'None'}")
if fits_df is not None:
    print(f"\nSample of fits_df:")
    print(fits_df.head())

## Figure 4: Sigmoidal Transitions in Cluster Probability

Fitting sigmoid functions to cluster probability curves reveals how quickly animals transition between neural response patterns across trials during sodium appetite states.

In [ ]:
# Verify sigmoid function is available
def sigmoid(x, A, L, x0, k):
    """4-parameter logistic function (matches assembly script)."""
    return A + (L - A) / (1 + np.exp(-k * (x - x0)))

# Check what's in fits_df from assembly
print(f"Fitted transitions available: {len(fits_df)} animals")
print(f"Columns in fits_df: {fits_df.columns.tolist()}")
print(f"\nSample of fits_df:")
print(fits_df.head())

### 4A. Cluster Probability by Condition

In [ ]:
# fits_df already contains only deplete + 45NaCl animals from assembly script
if fits_df is not None and len(fits_df) > 0:
    print(f"Loaded {len(fits_df)} deplete + 45NaCl animals with sigmoid fits")
    print(f"Columns in fits_df: {fits_df.columns.tolist()}")
    
    print(f"\nTransition Parameters Summary (Deplete + 45NaCl):")
    print(f"  x0 (transition point): {fits_df['x0_orig'].mean():.1f} ± {fits_df['x0_orig'].std():.1f} trials")
    print(f"  k (steepness):         {fits_df['k'].mean():.3f} ± {fits_df['k'].std():.3f}")
    print(f"  R² (fit quality):      {fits_df['r_squared'].mean():.3f} ± {fits_df['r_squared'].std():.3f}")
    
    print(f"\nIndividual animal fits:")
    for idx, row in fits_df.iterrows():
        animal_id = row.get('id', row.get('animal_id', f'animal_{idx}'))
        print(f"  {animal_id:5s} | x0={row['x0_orig']:6.1f}, k={row['k']:6.2f}, R²={row['r_squared']:5.3f}")
else:
    print("Warning: fits_df is empty or None - transition fits not available in assembled data")

### 4B. Representative Transition Curves

In [ ]:
# Plot representative sigmoid fits for deplete + 45NaCl animals
f, ax = plt.subplots(figsize=(3, 2.5))

# Get up to 8 animals with best fit quality
best_fits = fits_df.nlargest(8, 'r_squared')

# Create red-orange shades (one per rat)
rat_colors = plt.cm.OrRd(np.linspace(0.45, 0.9, len(best_fits)))

for idx, (_, row) in enumerate(best_fits.iterrows()):
    animal_id = row.get('id', row.get('animal_id', f'animal_{idx}'))
    rat_color = rat_colors[idx]
    
    # Get trials for this animal from x_array
    subset = x_array.query("id == @animal_id & condition == 'deplete' & infusiontype == '45NaCl'").sort_values('trial')
    trial_nums = subset.trial.values
    
    # Calculate cluster probability (cluster 0 = 1, else 0)
    cluster_membership = (subset.cluster_photo.values == 0).astype(int)
    
    # Smooth with rolling window
    window_size = min(5, len(cluster_membership))
    cluster_prob = pd.Series(cluster_membership).rolling(window=window_size, center=True).mean().values
    
    # Plot actual probability
    ax.plot(trial_nums, cluster_prob, 'o-', alpha=0.15, markersize=4, color=rat_color)
    
    # Plot fitted sigmoid using stored parameters
    y_fit = sigmoid(trial_nums, row['A'], row['L'], row['x0_orig'], row['k'])
    ax.plot(trial_nums, y_fit, '-', linewidth=1.5, alpha=0.35, color=rat_color)
    
    ax.scatter(row['x0_orig'], 1.1, marker='|', clip_on=False, color=rat_color)
    

ax.set_xlabel('Trial Number', fontsize=10)
ax.set_ylabel('Cluster 0 Probability', fontsize=10)
ax.text(28, 1.1, "Transition points", va="center")
# ax.set_title('Cluster Transitions (Deplete + 45NaCl)', fontsize=11)
ax.set_ylim([-0.1, 1.1])
sns.despine(ax=ax)

if SAVE_FIGS:
    save_figure_atomic(f, "fig4b_representative_transitions", FIGSFOLDER)

In [ ]:
# Logistic fits for raw cluster assignments (binary/inverted cluster_photo)
df_dep_45 = x_array.query("condition == 'deplete' & infusiontype == '45NaCl'").copy()
all_fits = []

f, ax = plt.subplots(figsize=(6,4))

for rat in df_dep_45.id.unique():
    sig = df_dep_45.loc[df_dep_45.id == rat, 'cluster_photo'].to_numpy()
    # Make binary: invert if needed (original code inverted)
    y = np.logical_not(sig).astype(int)
    x = np.arange(len(y), dtype=float)

    fit = fit_logistic_per_series(y, x=x, prefer_4p=True, direction='decreasing')
    all_fits.append({ 'id': rat, **fit['params'], 'model': fit['model'], 'x0_orig': fit['x0_orig'], 'success': fit['success'], 'note': fit['note'] })

    if fit["success"] and fit['x0_orig'] > 0 and fit['x0_orig'] < len(y):
        ax.plot(x, fit["y_hat"], color=colors[2], alpha=0.5, linestyle="--")

fits_df = pd.DataFrame(all_fits)
fits_df = fits_df.query("success == True and x0_orig > 0").copy()

x0 = fits_df['x0_orig'].to_list()
ax.plot(x0, [1.1]*len(x0), marker="o", linestyle="None", color=colors[2], alpha=0.5, clip_on=False)
ax.text(np.max(x0)+2, 1.1, "Transition points", ha="left", va="center", fontsize=10, color=colors[2])

ax.plot([np.mean(x0), np.mean(x0)], [1.05, 1.15], color=colors[2], linestyle="--", alpha=0.5, clip_on=False)
ax.text(np.mean(x0), 1.16, f"Mean=trial {int(np.mean(x0))}", ha="center", va="bottom", fontsize=10, color=colors[2])

sns.despine(ax=ax, offset=5)
ax.set_xlabel("Trial Number")
ax.set_ylabel("Probability of Cluster 1")

ax.set_yticks([0, 0.5, 1])
ax.set_ylim([-0.02, 1.1])

if savefigs:
    f.savefig(FIGSFOLDER / "logistic_fits_45NaCl.pdf", dpi=600, transparent=True)
    
fits_df_cluster_raw = fits_df
        
# print(fits_df)

### 4C. Transition Parameters Summary

In [ ]:
# Summary plots of transition parameters for deplete + 45NaCl animals
f, axes = plt.subplots(1, 2, figsize=(4, 2.5))

x0_data = fits_df['x0_orig'].values
k_data = fits_df['k'].values
animal_labels = fits_df['id'].values if 'id' in fits_df.columns else fits_df['animal_id'].values

# Plot 1: Transition point (x0)
ax = axes[0]
ax.scatter(np.arange(len(x0_data)), x0_data, color=colors[2], s=80, alpha=0.6, edgecolor='k', linewidth=1.5)
ax.axhline(x0_data.mean(), color=colors[2], linestyle='--', linewidth=2.5, alpha=0.7, 
           label=f"Mean: {x0_data.mean():.1f}±{x0_data.std():.1f}")
ax.fill_between([-0.5, len(x0_data)-0.5], 
                 x0_data.mean() - x0_data.std(),
                 x0_data.mean() + x0_data.std(), 
                 color=colors[2], alpha=0.2)
ax.set_xlabel('Animal', fontsize=10)
ax.set_ylabel('Transition Point (Trial #)', fontsize=10)
ax.set_title('Transition Midpoint (x0_orig)', fontsize=11)
ax.set_xticks(range(len(x0_data)))
ax.set_xticklabels(animal_labels, rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=9)
sns.despine(ax=ax)

# Plot 2: Steepness (k)
ax = axes[1]
ax.scatter(np.arange(len(k_data)), k_data, color=colors[2], s=80, alpha=0.6, edgecolor='k', linewidth=1.5)
ax.axhline(k_data.mean(), color=colors[2], linestyle='--', linewidth=2.5, alpha=0.7, 
           label=f"Mean: {k_data.mean():.3f}±{k_data.std():.3f}")
ax.fill_between([-0.5, len(k_data)-0.5],
                 k_data.mean() - k_data.std(),
                 k_data.mean() + k_data.std(),
                 color=colors[2], alpha=0.2)
ax.set_xlabel('Animal', fontsize=10)
ax.set_ylabel('Steepness (k)', fontsize=10)
ax.set_title('Transition Rate (k)', fontsize=11)
ax.set_xticks(range(len(k_data)))
ax.set_xticklabels(animal_labels, rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=9)
sns.despine(ax=ax)

plt.tight_layout()
if SAVE_FIGS:
    save_figure_atomic(f, "fig4c_transition_parameters", FIGSFOLDER)
plt.show()

## Save Results

In [ ]:
# Export deplete + 45NaCl transition fits for reference
fits_df.to_csv(RESULTSFOLDER / "transition_sigmoid_fits_deplete_45NaCl.csv", index=False)
print(f"Exported transition fits to {RESULTSFOLDER / 'transition_sigmoid_fits_deplete_45NaCl.csv'}")
print(f"\nFigure 4 generation complete!")